# Hyperparameter Tuning — all tunable models

Same searches as `python main.py --tune`: 5-fold CV on the **training split only** (scaling and the log1p target re-fit inside every fold). Ridge/Lasso/Elastic Net use an exhaustive grid; tree and boosting models use `RandomizedSearchCV` over capacity + regularisation parameters. Validation and test are never seen. Best parameters go to `reports/tuned_best_params.json` (read by `final_selection.ipynb`).

Mean/median baselines and plain Linear Regression have no hyperparameters to tune.

In [ ]:
import sys, pathlib, warnings, time, json
sys.path.insert(0, str(pathlib.Path.cwd().parent))
warnings.filterwarnings('ignore')
import importlib.util, pandas as pd
from src.preprocessing import prepare_data
from src.models import display_name, needs_package
from src.tuning import tune, best_params, best_cv_rmse, save_tuned_params, REG_GRIDS, RANDOM_SPACES, N_ITER
from src import config

In [ ]:
data = prepare_data()
keys = [k for k in list(REG_GRIDS) + list(RANDOM_SPACES)
        if not needs_package(k) or importlib.util.find_spec(needs_package(k))]
print('Tuning:', keys); print('Randomized budgets (candidates):', N_ITER)

In [ ]:
rows = []
for key in keys:
    t0 = time.perf_counter(); s = tune(key, data); p = best_params(s)
    save_tuned_params({key: p})
    rows.append({'model': display_name(key), 'key': key, 'cv_rmse': round(best_cv_rmse(s)),
                 'candidates': len(s.cv_results_['params']), 'seconds': round(time.perf_counter() - t0, 1),
                 'params': json.dumps(p, default=str)})
    print(f"{key:18s} CV RMSE Rs{best_cv_rmse(s):,.0f}  ({rows[-1]['seconds']}s)")
summary = pd.DataFrame(rows).sort_values('cv_rmse')
(config.REPORTS/'tuning').mkdir(exist_ok=True)
summary.to_csv(config.REPORTS/'tuning'/'hp_tuning_notebook_summary.csv', index=False)
summary

Next: `python scripts/robustness_check.py` (repeated CV + test bootstrap) and then `final_selection.ipynb`.